In [1]:
import glob
import platform

import pandas as pd

try:
    import torch
except ModuleNotFoundError:
    import sys, subprocess, importlib
    print("PyTorch not installed. Installing via pip (this may take several minutes)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch", "torchvision", "torchaudio"])
    importlib.invalidate_caches()
    import torch

import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

In [2]:
# Check the operating system - file location is different if its Windows or OS
if platform.system() == 'Windows':
    path = "G:/My Drive/EarthEngineData"
else:
    # For Holden's Mac
    path = "/Users/holden/Personal Projects/flame-flame-fruit/FireData"

files = glob.glob(path + "/*.csv")

In [3]:
# Throw all the files into one large pandas dataframe (this took me 22m to run btw)
df_list = []
for file in files:
    # Best practice is probably to put this all in a try catch but it worked for me for now...
    df = pd.read_csv(file)
    
    # Only add some of the days with no fire, since with too many it will skew predictions (since fire is rare)
    no_fires = df[df['T21_max'] == 0].sample(frac=0.0013413685916579098, random_state=42) #choosing same ratio as RF model
    # Every day with fire
    fires = df[df['T21_max'] > 0]
    
    both = pd.concat([fires, no_fires])
    df_list.append(both)

#Combine all the dataframes into one
final_df = pd.concat(df_list, ignore_index=True) #ignore_index to reset the row numbers on each list
print("final dataset size: ", final_df.shape)

final dataset size:  (166105, 30)


In [4]:
# Parse grid cell x,y indices out of system:index (format: YYYYMMDD_x,y)
# These represent which 4x4km grid cell in Colorado the row belongs to
final_df[['grid_x', 'grid_y']] = final_df['system:index'].str.split('_').str[1].str.split(',', expand=True).astype(int)

# Extracts the month number from the date (1-12) and adds it as a new column
final_df['month'] = pd.to_datetime(final_df['date']).dt.month

# Extracts year from the date and adds it as a new column
final_df['year'] = pd.to_datetime(final_df['date']).dt.year

# Creates a 0/1 column in case there is a fire
final_df['fire'] = (final_df['T21_max'] > 0).astype(int)

In [5]:
# This just helps visualize the data frame's structure
print(final_df.columns.tolist())
final_df.head(5)

['system:index', 'T21_max', 'T21_mean', 'T21_stdDev', 'aspect_max', 'aspect_mean', 'aspect_stdDev', 'date', 'elevation_max', 'elevation_mean', 'elevation_stdDev', 'erc_max', 'erc_mean', 'erc_stdDev', 'pr_max', 'pr_mean', 'pr_stdDev', 'rmin_max', 'rmin_mean', 'rmin_stdDev', 'slope_max', 'slope_mean', 'slope_stdDev', 'tmmx_max', 'tmmx_mean', 'tmmx_stdDev', 'vs_max', 'vs_mean', 'vs_stdDev', '.geo', 'grid_x', 'grid_y', 'month', 'year', 'fire']


,system:index,T21_max,T21_mean,T21_stdDev,aspect_max,aspect_mean,aspect_stdDev,date,elevation_max,elevation_mean,...,tmmx_stdDev,vs_max,vs_mean,vs_stdDev,.geo,grid_x,grid_y,month,year,fire
0,"20090106_119,1109",323.500000,217.290663,120.561332,109.0,103.590361,8.552128,2009-01-06,1719,1634.042169,...,0.890324,11.414818,9.866028,0.766249,"{""type"":""MultiPoint"",""coordinates"":[]}",119,1109,1,2009,1
1,"20090106_119,1110",323.500000,31.087087,120.561332,111.0,97.243243,9.154537,2009-01-06,1751,1667.360360,...,1.030267,10.506145,9.062752,0.698769,"{""type"":""MultiPoint"",""coordinates"":[]}",119,1110,1,2009,1
2,"20090112_137,1057",317.700012,31.492129,118.399804,46.0,26.851312,9.231588,2009-01-12,1458,1427.527697,...,0.055919,3.610772,3.552354,0.034750,"{""type"":""MultiPoint"",""coordinates"":[]}",137,1057,1,2009,1
3,"20090112_138,1057",317.700012,122.948921,137.568141,76.0,32.306502,22.015619,2009-01-12,1449,1421.492260,...,0.050070,3.635362,3.595614,0.024742,"{""type"":""MultiPoint"",""coordinates"":[]}",138,1057,1,2009,1
4,"20090112_137,1058",317.700012,19.856251,118.399804,102.0,75.860119,32.361242,2009-01-12,1434,1401.470238,...,0.137627,3.566329,3.494493,0.041712,"{""type"":""MultiPoint"",""coordinates"":[]}",137,1058,1,2009,1


In [6]:
# Split the data into training and testing sets
train_df = final_df[final_df['year'] < 2021]  # train on data before 2021
test_df = final_df[final_df['year'] >= 2021]  # test on data from 2021 and after

# Separate them into inputs and outputs
col_drop = ['fire', 'system:index', 'date', '.geo', 'T21_max', 'T21_mean', 'T21_stdDev']
X_train = train_df.drop(columns=col_drop)
X_test = test_df.drop(columns=col_drop)
y_train = train_df['fire']
y_test = test_df['fire']

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)
print("Fire rate in train:", y_train.mean().round(3))
print("Fire rate in test: ", y_test.mean().round(3))

Training set size: (120584, 28)
Test set size: (45521, 28)
Fire rate in train: 0.096
Fire rate in test:  0.079


In [7]:
# Standardize features - neural networks train much better with normalized inputs
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # use the same scaler fitted on training data

In [8]:
# Convert numpy arrays to PyTorch tensors
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).to(device)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).to(device)

# Wrap in a DataLoader for batched training
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)

Using device: cpu


In [9]:
# Define a simple feedforward neural network for binary classification
class FireNet(nn.Module):
    def __init__(self, input_dim):
        super(FireNet, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()  # outputs a probability between 0 and 1
        )

    def forward(self, x):
        return self.network(x).squeeze(1)

input_dim = X_train_scaled.shape[1]
model = FireNet(input_dim).to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

FireNet(
  (network): Sequential(
    (0): Linear(in_features=28, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=32, bias=True)
    (7): ReLU()
    (8): Linear(in_features=32, out_features=1, bias=True)
    (9): Sigmoid()
  )
)

Total parameters: 14,081


In [10]:
# Class weight to address imbalance: fire days are rare so we penalize missing them more
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(device)
print(f"pos_weight (penalty for missing a fire): {pos_weight.item():.2f}x")

# BCEWithLogitsLoss is numerically more stable, but we use BCE here since we already have Sigmoid
# Using pos_weight mirrors RandomForest's class_weight='balanced'
criterion = nn.BCELoss()

# Adam optimizer - adaptive learning rates per parameter, generally works well out of the box
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

pos_weight (penalty for missing a fire): 9.46x


In [13]:
import numpy as np
import torch

print("X_train has fillna lines applied? NaN in X_train:", int(X_train.isna().sum().sum()))
print("NaN in X_train_tensor:", torch.isnan(X_train_tensor).sum().item())
print("NaN/inf in X_train_scaled:", int(np.isnan(X_train_scaled).sum()), int(np.isinf(X_train_scaled).sum()))

X_train has fillna lines applied? NaN in X_train: 12294
NaN in X_train_tensor: 12294
NaN/inf in X_train_scaled: 12294 0


In [14]:
import torch

In [18]:
# Training loop
num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        preds = model(X_batch)

        # Apply per-sample weighting to handle class imbalance
        weights = torch.where(y_batch == 1, pos_weight.squeeze(), torch.ones(1).to(device))
        loss = (nn.BCELoss(reduction='none')(preds, y_batch) * weights).mean()

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1:02d}/{num_epochs}]  Loss: {avg_loss:.4f}")

print("Training complete.")

Epoch [01/20]  Loss: 1.0127
Epoch [05/20]  Loss: 0.8631
Epoch [10/20]  Loss: 0.8138
Epoch [15/20]  Loss: 0.7819
Epoch [20/20]  Loss: 0.7536
Training complete.


In [23]:
# Evaluate on the test set
model.eval()
with torch.no_grad():
    probs = model(X_test_tensor).cpu().numpy()

threshold = 0.6  # same threshold as the Random Forest model
predictions = (probs >= threshold).astype(int)

print(classification_report(y_test.values, predictions))

              precision    recall  f1-score   support

           0       0.92      1.00      0.96     41932
           1       0.00      0.00      0.00      3589

    accuracy                           0.92     45521
   macro avg       0.46      0.50      0.48     45521
weighted avg       0.85      0.92      0.88     45521



/opt/miniconda3/envs/brain3d/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/envs/brain3d/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/envs/brain3d/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", 

In [20]:
import torch, numpy as np
print("final_df exists:", 'final_df' in dir())
try:
    print("X_train_tensor NaN:", torch.isnan(X_train_tensor).sum().item())
except Exception as e:
    print("X_train_tensor:", e)
try:
    print("X_train NaN:", int(X_train.isna().sum().sum()))
except Exception as e:
    print("X_train:", e)
try:
    print("model output NaN:", torch.isnan(model(X_train_tensor[:500])).sum().item())
except Exception as e:
    print("model:", e)

final_df exists: True
X_train_tensor NaN: 0
X_train NaN: 0
model output NaN: 0


In [21]:
import numpy as np, torch
train_df = final_df[final_df['year'] < 2021]
test_df  = final_df[final_df['year'] >= 2021]
col_drop = ['fire','system:index','date','.geo','T21_max','T21_mean','T21_stdDev']
X_train = train_df.drop(columns=col_drop).replace([np.inf,-np.inf], np.nan)
X_test  = test_df.drop(columns=col_drop).replace([np.inf,-np.inf], np.nan)
means = X_train.mean()
X_train = X_train.fillna(means).fillna(0)
X_test  = X_test.fillna(means).fillna(0)
y_train = train_df['fire']; y_test = test_df['fire']

scaler = StandardScaler()
X_train_scaled = np.nan_to_num(scaler.fit_transform(X_train))
X_test_scaled  = np.nan_to_num(scaler.transform(X_test))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).to(device)
X_test_tensor  = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)
y_test_tensor  = torch.tensor(y_test.values, dtype=torch.float32).to(device)
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=1024, shuffle=True)

model = FireNet(X_train_scaled.shape[1]).to(device)
n_neg = (y_train==0).sum(); n_pos = (y_train==1).sum()
pos_weight = torch.tensor([n_neg/n_pos], dtype=torch.float32).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

print("NaN in X_train now:", int(X_train.isna().sum().sum()))
print("NaN in X_train_tensor now:", torch.isnan(X_train_tensor).sum().item())
print("NaN in model output now:", torch.isnan(model(X_train_tensor[:500])).sum().item())

NaN in X_train now: 0
NaN in X_train_tensor now: 0
NaN in model output now: 0
